## Transmission
This notebook demonstrates how to transmit dual-polarisation pulseshaped sequences using tremor-waveplate-toolbox.
We start by importing the Transmitter class:

In [ ]:
from tremor_waveplate_toolbox import Transmitter

Now, let's specify the transmitter parameters.
Parameters can be defined in code or stored in .ini files, the first of which is demonstrated below.

In [ ]:
from configparser import ConfigParser

# Define parameters in code
parameters = ConfigParser()
parameters['TRANSCEIVER'] = {
    'constellation':    'QPSK',  # The symbol constellation to use
    'power':            '2',     # Transmission power in dBm
    'baud_rate':        '1e6',   # Baud rate in symbols / s
    'pulse':            'RRCOS', # Pulseshape, can be SINC or RRCOS, or define your own using the Pulse class
    'pulse_parameter':  '0.5',   # Parameter to pass to the pulse constructor. For a RRCOS pulse, this is the rolloff factor
    'upsample_factor':  '4',     # Samples per symbol
    'filter':           'RRCOS', # Antialiasing (matched) filter
    'filter_parameter': '0.5'    # Same
}

parameters['SIGNAL'] = {
    'batch_size':   '1', # The number of signals to transmit simultaneously
    'symbol_count': '20' # The number of symbols to transmit per signal
}

We create and use a Transmitter as follows:

In [ ]:
Tx = Transmitter(parameters) # Create a transmitter with the defined parameters

_, signal = Tx.transmit_random((
    parameters.getint('SIGNAL', 'batch_size'),
    int(parameters.getfloat('SIGNAL', 'symbol_count'))
)) # Randomly draw a sequence of symbols, and form a pulseshaped signal

Let's plot the result

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2)

axes[0, 0].set_title("x polarisation")
axes[0, 0].set_ylabel("Real amplitude")
axes[0, 0].set_ylim([-0.04, 0.04])
axes[0, 0].grid()
axes[0, 0].plot(signal.time * 1e6, signal.samples_time[0, 0, :, 0].real)

axes[1, 0].set_xlabel("t [us]")
axes[1, 0].set_ylabel("Imaginary amplitude")
axes[1, 0].set_ylim([-0.04, 0.04])
axes[1, 0].grid()
axes[1, 0].plot(signal.time * 1e6, signal.samples_time[0, 0, :, 0].imag)

axes[0, 1].set_title("y polarisation")
axes[0, 1].set_ylim([-0.04, 0.04])
axes[0, 1].grid()
axes[0, 1].plot(signal.time * 1e6, signal.samples_time[0, 0, :, 1].real)

axes[1, 1].set_xlabel("t [us]")
axes[1, 1].set_ylim([-0.04, 0.04])
axes[1, 1].grid()
axes[1, 1].plot(signal.time * 1e6, signal.samples_time[0, 0, :, 1].imag)

fig.tight_layout()

plt.show()

You can also pulseshape a predefined symbol sequence:

In [ ]:
from tremor_waveplate_toolbox import QAM16

symbols = QAM16((
    parameters.getint('SIGNAL', 'batch_size'),
    int(parameters.getfloat('SIGNAL', 'symbol_count')),
    2
))
symbols, signal = Tx(symbols)

# Plot the result
fig, axes = plt.subplots(2, 2)

axes[0, 0].set_title("x polarisation")
axes[0, 0].set_ylabel("Real amplitude")
axes[0, 0].set_ylim([-0.04, 0.04])
axes[0, 0].grid()
axes[0, 0].plot(signal.time * 1e6, signal.samples_time[0, 0, :, 0].real)

axes[1, 0].set_xlabel("t [us]")
axes[1, 0].set_ylabel("Imaginary amplitude")
axes[1, 0].set_ylim([-0.04, 0.04])
axes[1, 0].grid()
axes[1, 0].plot(signal.time * 1e6, signal.samples_time[0, 0, :, 0].imag)

axes[0, 1].set_title("y polarisation")
axes[0, 1].set_ylim([-0.04, 0.04])
axes[0, 1].grid()
axes[0, 1].plot(signal.time * 1e6, signal.samples_time[0, 0, :, 1].real)

axes[1, 1].set_xlabel("t [us]")
axes[1, 1].set_ylim([-0.04, 0.04])
axes[1, 1].grid()
axes[1, 1].plot(signal.time * 1e6, signal.samples_time[0, 0, :, 1].imag)

fig.tight_layout()

plt.show()

Finally, let's use a receiver to matched-filter and downsample back to symbol space:

In [ ]:
from tremor_waveplate_toolbox import Receiver

receiver = Receiver(parameters)
symbols_received = receiver(signal)

# Plot the sent and received symbols
fig, axes = plt.subplots(2, 2)

axes[0, 0].set_title("x polarisation")
axes[0, 0].set_ylabel("Real amplitude")
axes[0, 0].set_ylim([-1.1, 1.1])
axes[0, 0].grid()
axes[0, 0].stem(symbols.time * 1e6, symbols.samples_time[0, 0, :, 0].real, markerfmt = 'bo')
axes[0, 0].stem(symbols_received.time * 1e6, symbols_received.samples_time[0, 0, :, 0].real, markerfmt = 'r.')

axes[1, 0].set_xlabel("t [us]")
axes[1, 0].set_ylabel("Imaginary amplitude")
axes[1, 0].set_ylim([-1.1, 1.1])
axes[1, 0].grid()
axes[1, 0].stem(symbols.time * 1e6, symbols.samples_time[0, 0, :, 0].imag, markerfmt = 'bo')
axes[1, 0].stem(symbols_received.time * 1e6, symbols_received.samples_time[0, 0, :, 0].imag, markerfmt = 'r.')

axes[0, 1].set_title("y polarisation")
axes[0, 1].set_ylim([-1.1, 1.1])
axes[0, 1].grid()
axes[0, 1].stem(symbols.time * 1e6, symbols.samples_time[0, 0, :, 1].real, markerfmt = 'bo')
axes[0, 1].stem(symbols_received.time * 1e6, symbols_received.samples_time[0, 0, :, 1].real, markerfmt = 'r.')

axes[1, 1].set_xlabel("t [us]")
axes[1, 1].set_ylim([-1.1, 1.1])
axes[1, 1].grid()
axes[1, 1].stem(symbols.time * 1e6, symbols.samples_time[0, 0, :, 1].imag, markerfmt = 'bo')
axes[1, 1].stem(symbols_received.time * 1e6, symbols_received.samples_time[0, 0, :, 1].imag, markerfmt = 'r.')

fig.legend(['Transmitted', 'Received'])
fig.tight_layout()

plt.show()